# Analyse et Prédiction de la GoldsteinScale (GDELT Dataset)
Ce script utilise un GradientBoostingRegressor pour prédire l'impact théorique 
d'un événement sur la stabilité du pays.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

### Configuration des variables

In [28]:
TARGET = "GoldsteinScale"
NUMERIC_FEATURES = [
    "IsRootEvent", 
    # "QuadClass", 
    "NumMentions", "NumSources", 
    "NumArticles", "AvgTone", "Actor1Geo_Type", "Actor2Geo_Type", 
    "ActionGeo_Type", "Actor1Geo_Lat", "Actor1Geo_Long", 
    "Actor2Geo_Lat", "Actor2Geo_Long", "ActionGeo_Lat", "ActionGeo_Long"
]
CATEGORICAL_FEATURES = [
    "Actor1CountryCode", "Actor2CountryCode", 
    # "EventRootCode", "EventBaseCode", 
    "Actor1Type1Code", "Actor2Type1Code", "ActionGeo_CountryCode"
]

# Les features commentés suffisent à elles seules à prédire parfaitement la GoldenSteinScale (potentielle fuite de données)

### Fonctions de Traitement

In [32]:
def prepare_features(df):
    """Nettoie et encode les donnees pour le modele."""
    df = df.copy()
    
    # Suppression des lignes sans cible
    df = df.dropna(subset=[TARGET])
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.dropna(subset=[TARGET])
    
    # Features numeriques : conversion et imputation par mediane
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())
    
    # Features categorielles : LabelEncoding et imputation par UNKNOWN
    le = LabelEncoder()
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].fillna("UNKNOWN").astype(str)
        df[col] = le.fit_transform(df[col])
    
    # Feature engineering : extraction de la date depuis SQLDATE (YYYYMMDD)
    if "SQLDATE" in df.columns:
        df["SQLDATE_INT"] = pd.to_numeric(df["SQLDATE"], errors="coerce").fillna(0).astype(int)
        df["Month"] = (df["SQLDATE_INT"] % 10000) // 100
        df["DayOfYear"] = df["SQLDATE_INT"] % 100
    else:
        df["Month"] = 0
        df["DayOfYear"] = 0

    all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["Month", "DayOfYear"]
    X = df[all_features]
    y = df[TARGET]
    return X, y

def train_model(X, y):
    """Entraine le modele et affiche les performances."""
    print("Split train/test (80/20)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"   Train: {len(X_train):,} | Test: {len(X_test):,}\n")
    print("Entrainement du modele GradientBoostingRegressor...")
    
    model = GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        random_state=42,
        verbose=0
    )
    model.fit(X_train, y_train)

    # Evaluation
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\nRESULTATS SUR LE JEU DE TEST")
    print("-" * 40)
    print(f"  MAE  (Mean Absolute Error) : {mae:.4f}")
    print(f"  R2   (Coefficient de det.) : {r2:.4f}")
    
    # Importance des features
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nTOP 10 FEATURES LES PLUS IMPORTANTES")
    print("-" * 40)
    for feat, imp in importances.head(10).items():
        bar = "#" * int(imp * 50)
        print(f"  {feat:<25} {imp:.4f}  {bar}")
    
    return model

### Exécution du Pipeline

In [33]:
CSV_PATH = "../data/clean/bq-results-last-12-months-clean.csv"

In [34]:
df_raw = pd.read_csv(CSV_PATH)

print("Preparation des donnees...")
X, y = prepare_features(df_raw)
print(f"Shape finale : X={X.shape}, y={y.shape}\n")

# Entrainement
model = train_model(X, y)

Preparation des donnees...
Shape finale : X=(23859, 21), y=(23859,)

Split train/test (80/20)...
   Train: 19,087 | Test: 4,772

Entrainement du modele GradientBoostingRegressor...

RESULTATS SUR LE JEU DE TEST
----------------------------------------
  MAE  (Mean Absolute Error) : 3.0585
  R2   (Coefficient de det.) : 0.2414

TOP 10 FEATURES LES PLUS IMPORTANTES
----------------------------------------
  AvgTone                   0.5190  #########################
  Actor1Type1Code           0.0777  ###
  Actor2Type1Code           0.0498  ##
  Actor2CountryCode         0.0419  ##
  Actor1Geo_Long            0.0412  ##
  Actor1Geo_Lat             0.0386  #
  Actor1CountryCode         0.0367  #
  Actor2Geo_Long            0.0334  #
  NumMentions               0.0284  #
  Actor2Geo_Lat             0.0239  #


In [25]:
import joblib

joblib.dump(model, '../models/gss_model.joblib')

['../models/gss_model.joblib']